In [1]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Memory: {props.total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU")

PyTorch: 2.11.0+cpu
CUDA available: False
No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU


In [4]:
%%writefile /content/net.py
"""
SmallCNN for 4-class CIFAR-10 subset.

Byte-identical to the local model/net.py. Any drift breaks model loading.
"""
import torch
import torch.nn as nn


CLASSES = ["airplane", "automobile", "bird", "cat"]
NUM_CLASSES = len(CLASSES)
CIFAR_MEAN = (0.5, 0.5, 0.5)
CIFAR_STD = (0.5, 0.5, 0.5)


class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

Writing /content/net.py


In [5]:
import sys
sys.path.insert(0, '/content')
from net import SmallCNN, CLASSES

m = SmallCNN()
print("Classes:", CLASSES)
print("Parameters:", sum(p.numel() for p in m.parameters()))

Classes: ['airplane', 'automobile', 'bird', 'cat']
Parameters: 618820


In [6]:
%%writefile /content/train.py
"""
Train SmallCNN on the 4-class CIFAR-10 subset.

Saves to /content/baseline.pt (Colab local filesystem).
Download to your machine with files.download() after training.
"""
import os
import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

sys.path.insert(0, '/content')
from net import SmallCNN, CLASSES


TARGET_LABELS = [0, 1, 2, 3]  # airplane, automobile, bird, cat
OUT_PATH = '/content/baseline.pt'


def get_subset_loader(train: bool, batch_size: int = 128):
    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    ds = datasets.CIFAR10(root='/content/data', train=train, download=True, transform=tfm)
    idx = [i for i, (_, y) in enumerate(ds) if y in TARGET_LABELS]
    return DataLoader(Subset(ds, idx), batch_size=batch_size, shuffle=train, num_workers=2)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


def train(epochs: int = 20, lr: float = 1e-3, batch_size: int = 128):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(42)

    train_loader = get_subset_loader(True, batch_size)
    test_loader = get_subset_loader(False, batch_size)

    print(f'Train batches: {len(train_loader)}')
    print(f'Test batches: {len(test_loader)}')

    model = SmallCNN().to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    best_acc = 0.0
    start = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            running_loss += loss.item()
        sched.step()

        acc = evaluate(model, test_loader, device)
        elapsed = time.time() - start
        print(f'epoch {epoch+1:02d}/{epochs} | '
              f'loss={running_loss/len(train_loader):.4f} | '
              f'eval_acc={acc:.4f} | t={elapsed:.0f}s')

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), OUT_PATH)
            print(f'  -> saved best (acc={acc:.4f})')

    print(f'\nFinal best: {best_acc:.4f}')
    print(f'Saved to: {OUT_PATH}')
    return model


if __name__ == '__main__':
    train(epochs=20)

Writing /content/train.py


In [7]:
%cd /content
!python train.py

/content
Device: cpu
100% 170M/170M [31:31<00:00, 90.2kB/s]
Train batches: 157
Test batches: 32
epoch 01/20 | loss=0.8303 | eval_acc=0.7495 | t=48s
  -> saved best (acc=0.7495)
epoch 02/20 | loss=0.6014 | eval_acc=0.7802 | t=95s
  -> saved best (acc=0.7802)
epoch 03/20 | loss=0.5157 | eval_acc=0.7885 | t=141s
  -> saved best (acc=0.7885)
epoch 04/20 | loss=0.4373 | eval_acc=0.7963 | t=188s
  -> saved best (acc=0.7963)
epoch 05/20 | loss=0.3883 | eval_acc=0.8285 | t=235s
  -> saved best (acc=0.8285)
epoch 06/20 | loss=0.3400 | eval_acc=0.8355 | t=283s
  -> saved best (acc=0.8355)
epoch 07/20 | loss=0.2897 | eval_acc=0.8340 | t=330s
epoch 08/20 | loss=0.2514 | eval_acc=0.8578 | t=377s
  -> saved best (acc=0.8578)
epoch 09/20 | loss=0.2163 | eval_acc=0.8552 | t=425s
epoch 10/20 | loss=0.1764 | eval_acc=0.8532 | t=473s
epoch 11/20 | loss=0.1454 | eval_acc=0.8590 | t=522s
  -> saved best (acc=0.8590)
epoch 12/20 | loss=0.1148 | eval_acc=0.8590 | t=570s
epoch 13/20 | loss=0.0928 | eval_acc=0

In [8]:
import torchvision
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import os

SAMPLES_DIR = '/content/samples'
os.makedirs(SAMPLES_DIR, exist_ok=True)

classes = ['airplane', 'automobile', 'bird', 'cat']
ds = torchvision.datasets.CIFAR10(root='/content/data', train=False, download=True, transform=transforms.ToTensor())

saved = {0: 0, 1: 0, 2: 0, 3: 0}
for img, label in ds:
    if label in saved and saved[label] < 5:
        np_img = (img.numpy().transpose(1, 2, 0) * 255).astype('uint8')
        Image.fromarray(np_img).save(f'{SAMPLES_DIR}/{classes[label]}_{saved[label]}.png')
        saved[label] += 1
    if all(v >= 5 for v in saved.values()):
        break

print(f'Saved to {SAMPLES_DIR}')
print(sorted(os.listdir(SAMPLES_DIR)))

Saved to /content/samples
['airplane_0.png', 'airplane_1.png', 'airplane_2.png', 'airplane_3.png', 'airplane_4.png', 'automobile_0.png', 'automobile_1.png', 'automobile_2.png', 'automobile_3.png', 'automobile_4.png', 'bird_0.png', 'bird_1.png', 'bird_2.png', 'bird_3.png', 'bird_4.png', 'cat_0.png', 'cat_1.png', 'cat_2.png', 'cat_3.png', 'cat_4.png']


In [9]:
import shutil
shutil.make_archive('/content/samples', 'zip', '/content/samples')
print('Created /content/samples.zip')

Created /content/samples.zip


In [10]:
from google.colab import files
files.download('/content/baseline.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from google.colab import files
files.download('/content/samples.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>